# Baseline CNN with training-only oversampling

Run `chest_xray_train_val_test_split.ipynb` first, including its duplicate-removal cell. This notebook loads the prepared `chest_xray_cnn_splits/{train,val,test}/{NORMAL,PNEUMONIA}` folders, then balances **only the training dataset** by repeating examples from the minority class, following `oversample_training_dataset` in `imbalance.ipynb`. Validation and test remain unchanged. The CNN architecture and final evaluation are the same baseline approach as the class-weighted notebook.

In [ ]:
%pip install -q tensorflow kagglehub numpy scikit-learn matplotlib

## Find prepared data and load splits

In [ ]:
from pathlib import Path
import kagglehub
import numpy as np
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"

download_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
candidate_dirs = [
    Path.cwd() / "chest_xray_cnn_splits",
    Path.cwd().parent / "chest_xray_cnn_splits",
    download_dir.parent / "chest_xray_cnn_splits",
]
data_dir = next(
    (path for path in candidate_dirs if all((path / split).is_dir() for split in ("train", "val", "test"))),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Prepared splits not found. Run chest_xray_train_val_test_split.ipynb "
        "through its split and duplicate-removal cells first. Checked: "
        + ", ".join(str(path) for path in candidate_dirs)
    )
print(f"Using prepared data from: {data_dir}")

def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        data_dir / split,
        image_size=IMAGE_SIZE,
        color_mode="grayscale",
        batch_size=BATCH_SIZE,
        label_mode="int",
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )

train_ds = load_split("train", shuffle=True)
val_ds = load_split("val", shuffle=False)
test_ds = load_split("test", shuffle=False)
class_names = train_ds.class_names
if class_names != ["NORMAL", "PNEUMONIA"]:
    raise ValueError(f"Expected classes ['NORMAL', 'PNEUMONIA']; got {class_names}")
if val_ds.class_names != class_names or test_ds.class_names != class_names:
    raise ValueError("Train, validation, and test folders must have matching classes.")
AUTOTUNE = tf.data.AUTOTUNE
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)
print("Class mapping:", {name: index for index, name in enumerate(class_names)})

## Oversample the training dataset

This applies the same approach as `imbalance.ipynb`: count sparse labels, repeat each class to match the largest class, shuffle, then batch. It does not create extra image files. Only `train_ds` is replaced; validation and test are not sampled or augmented.

In [ ]:
num_classes = len(class_names)
train_counts = np.zeros(num_classes, dtype=np.int64)
for _, labels in train_ds:
    train_counts += np.bincount(labels.numpy().reshape(-1), minlength=num_classes)
if np.any(train_counts == 0):
    missing = [class_names[i] for i, count in enumerate(train_counts) if count == 0]
    raise ValueError(f"Cannot oversample; training classes are absent: {missing}")

target_per_class = int(train_counts.max())
unbatched_train = train_ds.unbatch()
class_datasets = []
for class_id in range(num_classes):
    class_dataset = unbatched_train.filter(
        lambda _image, label, expected=class_id: tf.equal(
            tf.cast(tf.reshape(label, []), tf.int64), expected
        )
    )
    class_datasets.append(class_dataset.repeat().take(target_per_class))

balanced_train_ds = class_datasets[0]
for class_dataset in class_datasets[1:]:
    balanced_train_ds = balanced_train_ds.concatenate(class_dataset)
balanced_train_ds = balanced_train_ds.shuffle(
    buffer_size=min(num_classes * target_per_class, 1000),
    seed=SEED,
    reshuffle_each_iteration=True,
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Original training counts:")
for class_id, name in enumerate(class_names):
    print(f"  {name}: {train_counts[class_id]:,}")
print(f"Oversampled training count: {target_per_class:,} per class")

## Build and train baseline CNN

In [ ]:
tf.keras.utils.set_random_seed(SEED)
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 1)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ],
    name="baseline_cnn_oversampling",
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
history = model.fit(
    balanced_train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],
)

## Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
y_true = np.concatenate([labels.numpy().reshape(-1) for _, labels in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()